# Quantum walk circuits
$
\newcommand{\ket}[1]{\left|#1\right\rangle}
\newcommand{\bra}[1]{\left\langle #1\right|}
\newcommand{\braket}[2]{\left\langle #1 \middle| #2 \right\rangle}
\newcommand{\ketbra}[2]{\left|#1\right\rangle\!\left\langle #2\right|}
$

In [1]:
# allows us to have visibility on our package without installing it in editing mode
import sys;
if ".." not in sys.path: sys.path.append("..")

import itertools
import numpy as np

from qiskit.circuit import QuantumCircuit, Gate

from monaqa2.qiskit.utils_numpy import kron, ketbra, bra, ket
from monaqa2.qiskit.utils_qiskit import get_unitary

chiamare trial move $T$ e la transition matrix $P$

The quantum walk circuit is an operator acting on the Hilbert space $\mathcal{H}_a \otimes \mathcal{H}_b \otimes \mathcal{H}_c$, where $\mathcal{H}_a$ and $\mathcal{H}_b$ are two copies of the system register holding the MCMC state, and $\mathcal{H}_c$ is an auxiliary register called the coin. The walk operator has the form

$$
W = R_0 V^\dagger B^\dagger F B V.
$$

Here,

$$
V\ket{x}_a\ket{0}_b
=
\ket{x}_a
\sum_y \sqrt{P_{yx}}\ket{y}_b
$$

is the proposal unitary. The quantity $P_{yx}$ is nonzero when the move from state $x$ to state $y$ is allowed with nonzero proposal probability.

The Boltzmann coin is

$$
B\ket{x}_a\ket{y}_b\ket{0}_c
=
\ket{x}_a\ket{y}_b
\left(
\sqrt{A_{yx}}\ket{0\cdots 0}_c
+
\sqrt{1-A_{yx}}\ket{\bot_{xy}}_c
\right),
$$

where $A_{yx}$ is the probability of accepting the move from $x$ to $y$, determined by the energy difference between the two states and the inverse temperature. The state $\ket{\bot_{xy}}_c$ is orthogonal to $\ket{0\cdots 0}_c$.

The accept-path swap is

$$
\begin{align*}
F\ket{x}_a\ket{y}_b\ket{0\cdots 0}_c
&=
\ket{y}_a\ket{x}_b\ket{0\cdots 0}_c, \\
F\ket{x}_a\ket{y}_b\ket{\bot_{xy}}_c
&=
\ket{x}_a\ket{y}_b\ket{\bot_{xy}}_c.
\end{align*}
$$

Finally, $R_0$ acts as the identity on $\mathcal{H}_a$ and as a reflection about $\ket{0\cdots 0}_b\ket{0\cdots 0}_c$ on $\mathcal{H}_b \otimes \mathcal{H}_c$.

We expect three properties to hold.

First, $U = V^\dagger B^\dagger F B V$ is a block encoding of $X$, the discriminant matrix of the MCMC:

$$
(\mathbb{I}_a \otimes \bra{0}_b \otimes \bra{0}_c)
U
(\mathbb{I}_a \otimes \ket{0}_b \otimes \ket{0}_c)
=
X.
$$

For a reversible Markov chain with transition matrix $P$ and stationary distribution $\pi$, the discriminant has entries $X_{yx}=\sqrt{P_{xy}P_{yx}}$.

Second, the stationary distribution is encoded in the zero-ancilla invariant subspace. In particular, detailed balance implies that the coherent stationary state $\ket{\sqrt{\pi}}=\sum_x \sqrt{\pi_x}\ket{x}$ is a $+1$ eigenvector of $X$:

$$
X\ket{\sqrt{\pi}} = \ket{\sqrt{\pi}}.
$$

Therefore, because $U$ block-encodes $X$, the full state $\ket{\sqrt{\pi}}_a\ket{0}_b\ket{0}_c$ is a $+1$ eigenstate of $U$ with no leakage outside the zero subspace.

Third, if $\lambda \in \mathrm{spec}(X)$, then the corresponding walk eigenvalues are

$$
\exp(\pm i \arccos(\lambda)) \in \mathrm{spec}(W).
$$

## Template for all circuits

### Reflection gate

The reflection gate $R_0$ is in module:

In [2]:
from monaqa2.qiskit.reflection_gate import Reflection

There are two implementations available. 

The standard implementation has $O(n)$ depth and uses a multi-controlled NOT without any extra ancillas. As this is not the implementation considered in the paper, it is instatiated with the mocked_circuit=True parameter in the initialization, and is mainly used for tests in order to minimize the size of the system to simulate. 

We can check the behaviour matched the expectation by comparing the unitary associated with the quantum circuit with its linear algebraic equivalent. 

In [3]:
from monaqa2.qiskit.utils_numpy import kron, ketbra, bra, ket
from monaqa2.qiskit.utils_qiskit import get_unitary

n = 2 # number of spins in the system 
c = 2 # number of qubits in the coin subsystem

# create the expected unitary: I \otimes 2(|0..0><0..0| \otimes |0..0><0..0| - I
R = 2 * kron(ketbra(0, n=n), ketbra(0, n=c)) - np.eye(2**(n+c))
expected = kron(np.eye(2**n), R)

# instantiate the circuit then retrieve the associated unitary
refl = Reflection(n=n, coins=c, mocked_circuit=True)
qc = QuantumCircuit(refl.num_qubits)
qc.append(refl, range(refl.num_qubits))
actual = np.real_if_close(get_unitary(qc, big_endian=True))

err = np.linalg.norm(expected - actual, ord=2)
print(f"The error between the ideal unitary and the quantum circuit is {err:1.4e}")

The error between the ideal unitary and the quantum circuit is 7.7716e-16


To minimize the depth of this circuit, a $O(\log n)$-depth implementation is possible albeit at the expense of some auxiliary qubits. In this case, the multi-controlled NOT uses $n-1$ ancillas to create a balance tree of "AND" clauses holding partial results of the test whether thestate is $\ket{0\cdots 0}$. This is the default implementation. 

In [4]:
# instantiate the actual circuit
refl = Reflection(n=n, coins=c)
qc = QuantumCircuit(refl.num_qubits)
qc.append(refl, range(refl.num_qubits))

Here the layout property is telling us which qubits of this enlarge unitary are used as a workspace.

In [5]:
refl.layout

{'A': [0, 1],
 'B': [2, 3],
 'coins': [4, 5],
 'work': [6],
 'reflected_register': [2, 3, 4, 5]}

We get the unitary associated with the quantum circuit, then pick the portion related to the subsystem starting and ending with the workspace qubits in state $\ket{0\cdots 0}$. 

In [6]:
U = np.real_if_close(get_unitary(qc, big_endian=True))
n_work = len(refl.layout['work'])
actual = kron(np.eye(2**(n+n+c)), bra(0, n=n_work)) @ U @ kron(np.eye(2**(n+n+c)), ket(0, n=n_work))

err = np.linalg.norm(expected - actual, ord=2)
print(f"The error between the ideal unitary and the quantum circuit is {err:1.4e}")

The error between the ideal unitary and the quantum circuit is 2.2204e-16


### Accept path

The accept-path gate $F$ is in module:

In [7]:
from monaqa2.qiskit.accept_path_gate import AcceptPath
from monaqa2.qiskit.utils_numpy import swap

There are two implementations available. 

The standard implementation has $O(n)$ depth and uses a multi-controlled NOT without any extra ancillas, is used to control the SWAP operations. Again, this is not the implementation considered in the paper, it is instatiated with the mocked_circuit=True parameter in the initialization.

We can check the behaviour matched the expected one, 

$$F = \mathrm{SWAP}_{a, b} \otimes \ket{0\cdots 0}\!\bra{0\cdots 0} + \mathbb{I}_{a,b} \otimes \ket{\bot}\!\bra{\bot}$$

In [8]:
n = 2 # number of spins in the system 
c = 2 # number of qubits in the coin subsystem

# create the expected unitary: I \otimes 2(|0..0><0..0| \otimes |0..0><0..0| - I
coin_zero = ketbra(0, n=c)
coin_bot = np.eye(2**c) - coin_zero
F = kron(swap(n), coin_zero) + kron(np.eye(2**(2*n)), coin_bot)
expected = F

# instantiate the circuit then retrieve the associated unitary
ap = AcceptPath(n=n, coins=c, mocked_circuit=True)
qc = QuantumCircuit(ap.num_qubits)
qc.append(ap, range(ap.num_qubits))
actual = np.real_if_close(get_unitary(qc, big_endian=True))

err = np.linalg.norm(expected - actual, ord=2)
print(f"The error between the ideal unitary and the quantum circuit is {err:1.4e}")

The error between the ideal unitary and the quantum circuit is 1.6385e-15


The $O(\log n)$-depth based circuit allocates $n$ auxiliary qubits and proceeds as follows:
1. use a log-depth MCX (tree of reversible AND) to save the test of $0\cdots 0$ condition into a flag qubit, then clean the workspace;
2. with a balanced tree of CX copy the flag into other $n-1$ clean ancillas;
3. use all the $n$ ancillas containing the flag to run in parallel all the $n$ controlled-SWAP operator;
4. clean up.

In [9]:
# instantiate the circuit then retrieve the associated unitary
ap = AcceptPath(n=n, coins=c)
qc = QuantumCircuit(ap.num_qubits)
qc.append(ap, range(ap.num_qubits))

U = np.real_if_close(get_unitary(qc, big_endian=True))
n_work = len(ap.layout['work'])
actual = kron(np.eye(2**(n+n+c)), bra(0, n=n_work)) @ U @ kron(np.eye(2**(n+n+c)), ket(0, n=n_work))

err = np.linalg.norm(expected - actual, ord=2)
print(f"The error between the ideal unitary and the quantum circuit is {err:1.4e}")

The error between the ideal unitary and the quantum circuit is 0.0000e+00


### Boltzmann coin

The actual implementation of the Boltzmann coin in explained in details in 2_phase_arithmetic.ipynb. A mocked implementation based on a state preparation routine is used here for test purposes (albeit has $O(\exp n)$ gates, it uses no ancilla qubits and thus might be used to simulate the walk at small $n$).

In [10]:
from monaqa2.qiskit.glauber_stateprep_arithmetic_gate import GlauberStateprepArithmetic
from monaqa2.qiskit.utils_numpy import expected_glauber_amplitude

The circuit implements the unitary

$$
B:\ \ket{x}_a \ket{y}_b \ket{0}_c
\mapsto
\ket{x}_a \ket{y}_b
\left(
g(x,y)\ket{0}_c
+
\sqrt{1-g(x,y)^2}\ket{1}_c
\right).
$$

Here $g(x, y)$ is the generalized Glauber acceptance ($a \ge 1$):

$$g(x,y)=\sqrt{\left(\frac{1}{1+\exp^{\beta \Delta E(x,y)}}\right)^{1/a}}$$

For $a=\infty$ the rule matches the Metropolis-Hastings acceptance:

$$
g(x,y)
=
\sqrt{
\min(1,\exp^{-\beta \Delta E(x,y)})
}.
$$

We test it by constructing the full unitary matrix $U_{\mathrm{acc}}$ of the circuit. For every computational-basis pair $(x,y)$, we prepare the input state $\ket{x}_a\ket{y}_b\ket{0}_c$ and extract the clean-coin matrix element

$$
\bra{x}_a \bra{y}_b \bra{0}_c
U_{\mathrm{acc}}
\ket{x}_a \ket{y}_b \ket{0}_c.
$$

This value is compared against the analytically computed amplitude $g(x,y)$. The reported error is the squared amplitude error $|g(x,y)-\bra{x}_a\bra{y}_b\bra{0}_c U_{\mathrm{acc}}\ket{x}_a\ket{y}_b\ket{0}_c|^2$.

In [11]:
n = 2
c = 2

h = np.array([0.5, -0.25])
J = np.array([[0.0, 0.2], [0.2, 0.0]])

beta = 10.0
a = 10 # a = np.inf is the Metropolis-Hastings rule

glauber_arith = GlauberStateprepArithmetic(n, c, h, J, beta, a=a)

qc = QuantumCircuit(glauber_arith.num_qubits)
qc.append(glauber_arith, range(glauber_arith.num_qubits))
U = np.real_if_close(get_unitary(qc, big_endian=True))

for x, y in itertools.product(range(2**n), repeat=2):
    expected_amplitude = expected_glauber_amplitude(x, y, n, h, J, beta, a)

    ket_in = kron(ket(x, n), ket(y, n), ket(0, n=c))
    bra_out = kron(bra(x, n), bra(y, n), bra(0, n=c))
    actual_amplitude = np.real_if_close((bra_out @ U @ ket_in).item())

    err = np.abs(expected_amplitude - actual_amplitude) ** 2

    print(
        f"x={x:0{n}b}, y={y:0{n}b}, "
        f"expected={expected_amplitude:.6f}, "
        f"actual={actual_amplitude:.6f}, "
        f"err={err:.2e}"
    )

x=00, y=00, expected=0.965936, actual=0.965936, err=7.18e-26
x=00, y=01, expected=0.606529, actual=0.606529, err=2.84e-26
x=00, y=10, expected=1.000000, actual=1.000000, err=7.25e-26
x=00, y=11, expected=1.000000, actual=1.000000, err=7.19e-26
x=01, y=00, expected=0.999998, actual=0.999998, err=7.61e-26
x=01, y=01, expected=0.965936, actual=0.965936, err=7.11e-26
x=01, y=10, expected=1.000000, actual=1.000000, err=6.99e-26
x=01, y=11, expected=1.000000, actual=1.000000, err=7.16e-26
x=10, y=00, expected=0.000912, actual=0.000912, err=1.18e-29
x=10, y=01, expected=0.000553, actual=0.000553, err=1.13e-30
x=10, y=10, expected=0.965936, actual=0.965936, err=7.02e-26
x=10, y=11, expected=0.011109, actual=0.011109, err=8.81e-29
x=11, y=00, expected=0.082085, actual=0.082085, err=7.63e-28
x=11, y=01, expected=0.049787, actual=0.049787, err=3.87e-28
x=11, y=10, expected=1.000000, actual=1.000000, err=7.62e-26
x=11, y=11, expected=0.965936, actual=0.965936, err=7.20e-26


## Walk with uniform proposal

### Proposal unitary

The proposal unitary for the uniform move acts as

$$
V_{\mathrm{unif}}\ket{x}_a\ket{0}_b
=
\ket{x}_a
\sum_y \frac{1}{2^{n/2}}\ket{y}_b.
$$

For fixed input state $x$, the proposal amplitude for output state $y$ in the second register is obtained from the matrix element

$$
({}_a\bra{x}{}_b\bra{y}) \,
V_{\mathrm{unif}} \,
(\ket{x}_a\ket{0}_b)
=
\frac{1}{2^{n/2}}
=
\sqrt{P_{\mathrm{unif}}(y,x)}.
$$

Equivalently,

$$
\sum_x
(\bra{x}_a \otimes \mathbb{I}_b)
\,V_{\mathrm{unif}}\,
(\ket{x}\bra{x}_a \otimes \ket{0}_b)
=
\frac{1}{2^{n/2}}
\sum_{x,y}\ket{y}_b\bra{x}_a
=
\sqrt{P_{\mathrm{unif}}}.
$$

It is implemented by leaving untouched the register $\ket{\cdot}_a$ and applying hadamard gates on each qubit of the register $\ket{\cdot}_b$. In fact, in the uniform _move_ the next state does not depend on $x$ (the acceptance still does but that aspect is taken care by the boltzmann coin). 

In [12]:
from monaqa2.qiskit.proposal_uniform_gate import ProposalUniform
from monaqa2.mcmc.proposal import create_proposal_matrix_uniform

# Set the number of spins.
n = 2

# Build the ideal proposal-amplitude matrix.
P_unif = create_proposal_matrix_uniform(n)
expected = np.sqrt(P_unif)

# Build the quantum circuit implementing the uniform proposal.
v_unif = ProposalUniform(n)
qc = QuantumCircuit(v_unif.num_qubits)
qc.append(v_unif, range(v_unif.num_qubits))

# Extract the unitary matrix of the circuit.
V_unif = np.real_if_close(get_unitary(qc, big_endian=True))

# Contract the preserved 'a' register to recover sqrt(P_unif).
actual = sum(kron(bra(x, n), np.eye(2**n)) @ V_unif @ kron(ketbra(x, n), ket(0, n)) for x in range(2**n))
actual = np.real_if_close(actual)

# Compare the ideal and circuit-derived proposal amplitudes.
err = np.linalg.norm(expected - actual, ord=2)
print(f"The error between the ideal transformation and the quantum circuit is {err:1.4e}")

The error between the ideal transformation and the quantum circuit is 4.4409e-16


### Walk unitary

The walk operator with uniform move must satify three conditions:

* $R_0 W = V^\dagger B^\dagger F B V$ is a block encoding of $X$, i.e.
$$ (\mathbb{I}_a \otimes \bra{0}_b \otimes \bra{0}_c) U (\mathbb{I}_a \otimes \ket{0}_b \otimes \ket{0}_c) = X$$
* $R_0 W$ has $\ket{\sqrt{\pi}}_a\ket{0}_b\ket{0}_c$ as its own $+1$ eigenvector, i.e.
$$R_0 W\ket{\sqrt{\pi}}_a\ket{0}_b\ket{0}_c = (+1) \ket{\sqrt{\pi}}_a\ket{0}_b\ket{0}_c$$
* $\lambda \in \mathrm{spec}(X)$ implies
$$\exp(\pm i \arccos(\lambda)) \in \mathrm{spec}(W)$$

The following snippet instantiate the quantum circuit for $W$ and retrieve the unitary, then construct both the distriminant $X$ and the stationary distribution $\pi$, which will be used in the code just below. 

In [13]:
from monaqa2.qiskit.walk_uniform_gate import WalkUniform
from monaqa2.mcmc.transition import create_transition_matrix
from monaqa2.mcmc.distribution import get_gibbs_distribution_hJ

n = 3
h = np.array([0.9, -0.6, 0.35])
J = np.array([
    [0.0, -0.75, 0.40],
    [-0.75, 0.0, -0.55],
    [0.40, -0.55, 0.0],
])
beta = 12.0
a = 10
c = 1

# Build the classical uniform-proposal transition matrix and its discriminant.
P_unif = create_proposal_matrix_uniform(n)
Q = create_transition_matrix(P_unif, h, J, beta, a=a)
X = np.sqrt(Q * Q.T)

# Build the target Gibbs stationary state.
pi = get_gibbs_distribution_hJ(h, J, beta=beta)
sqrt_pi = np.sqrt(pi).reshape(-1, 1)

# Here mocked_circuit=True means we use the ancilla-free versions of Reflection and AcceptPath.
walk = WalkUniform(n, h, J, beta, eps=None, coin="stateprep", a=a, c=c, mocked_circuit=True)
qc = QuantumCircuit(walk.num_qubits)
qc.append(walk, range(walk.num_qubits))

# Extract the unitary matrix of the circuit.
W_unif = np.real_if_close(get_unitary(qc, big_endian=True))

Here we check that $W$ is a block encoding for $X$. Actually $R_0 W$ is naturally the block encoding here but the reflection does not change anything so we keep things easy. 

In [14]:
# Test 1: the zero-block of W equals the discriminant X.
left_zero = kron(np.eye(2**n), bra(0, n), bra(0, walk.coins), bra(0, walk.n_accept_work), bra(0, walk.n_reflection_work))
right_zero = kron(np.eye(2**n), ket(0, n), ket(0, walk.coins), ket(0, walk.n_accept_work), ket(0, walk.n_reflection_work))
actual_X = np.real_if_close(left_zero @ W_unif @ right_zero)

err_block = np.linalg.norm(X - actual_X, ord=2)
print(f"Test 1: block-encoding error ||X - <0|W|0>|| = {err_block:1.4e}")

Test 1: block-encoding error ||X - <0|W|0>|| = 3.3633e-13


Then we check that the coherent Gibbs state $\ket{\pi}\ket{0}\ket{0}$ is actually a $+1$ eigenstate for $W$. 

In [15]:
# Test 2: the coherent Gibbs state is a +1 eigenstate.
stationary_state = kron(sqrt_pi, ket(0, n), ket(0, walk.coins), ket(0, walk.n_accept_work), ket(0, walk.n_reflection_work))

err_stationary_X = np.linalg.norm(X @ sqrt_pi - sqrt_pi)
err_stationary_W = np.linalg.norm(W_unif @ stationary_state - stationary_state)

print(f"Test 2: discriminant stationary-state error ||X|sqrt(pi)> - |sqrt(pi)>|| = {err_stationary_X:1.4e}")
print(f"Test 2: walk stationary-state error ||W|sqrt(pi),0,0> - |sqrt(pi),0,0>|| = {err_stationary_W:1.4e}")

Test 2: discriminant stationary-state error ||X|sqrt(pi)> - |sqrt(pi)>|| = 1.5701e-16
Test 2: walk stationary-state error ||W|sqrt(pi),0,0> - |sqrt(pi),0,0>|| = 3.3640e-13


Finally we show that the spectrum of the walk is linked to the spectrum of $X$.

In [16]:
# Test 3: each lambda in spec(X) induces exp(+- i arccos(lambda)) in spec(W).
actual_walk_eigs = np.linalg.eigvals(W_unif)
expected_walk_eigs = np.ravel([[np.exp(1j * np.arccos(lam)), np.exp(-1j * np.arccos(lam))] 
                               for lam in np.clip(np.linalg.eigvalsh(X), -1.0, 1.0)])
err_spectral = max([np.min(np.abs(actual_walk_eigs - eig)) for eig in expected_walk_eigs])

print(f"Test 3: spectral-mapping error = {err_spectral:1.4e}")

Test 3: spectral-mapping error = 2.1073e-08


## Walk with local spin-flip proposal

### Proposal matrix

The local spin-flip proposal is implemented by preparing a weight-$k$ Dicke state, giving a uniform superposition over bitstrings $z \in \{0,1\}^n$ with $|z| = k$. The string $z$ marks the $k$ spins to flip. Applying CNOTs from this register to the current configuration $x$ gives the proposal $y = x \oplus z$, where $\oplus$ is bitwise XOR. Since $|z| = k$, the proposal satisfies $d_H(x,y) = k$.

Thus the proposal matrix is

$$
P_{(x \oplus z), x}  =
\begin{cases}
1 / \binom{n}{k}, & d_H(x,x \oplus z) = k\\
0, & \text{otherwise}.
\end{cases}
$$

The Dicke state preparation has depth ... and the CNOT layer has depth constant because...

In [17]:
from monaqa2.qiskit.proposal_local_gate import ProposalLocal
from monaqa2.mcmc.proposal import create_proposal_matrix_local

# Set the number of spins and the Hamming-weight move size.
n = 3
k = 2

# Build the ideal proposal-amplitude matrix for exactly-k local moves.
P_local = create_proposal_matrix_local(n, k=k)
expected = np.sqrt(P_local)

# Build the quantum circuit implementing the local proposal.
v_local = ProposalLocal(n, k)
qc = QuantumCircuit(v_local.num_qubits)
qc.append(v_local, range(v_local.num_qubits))

# Extract the unitary matrix of the circuit.
V_local = np.real_if_close(get_unitary(qc, big_endian=True))

# Contract the preserved 'a' register to recover sqrt(P_local).
actual = sum(kron(bra(x, n), np.eye(2**n)) @ V_local @ kron(ketbra(x, n), ket(0, n)) for x in range(2**n))
actual = np.real_if_close(actual)

# Compare the ideal and circuit-derived proposal amplitudes.
err = np.linalg.norm(expected - actual, ord=2)
print(f"The error between the ideal transformation and the quantum circuit is {err:1.4e}")

The error between the ideal transformation and the quantum circuit is 1.1102e-16


### Walk operator

In [18]:
from monaqa2.qiskit.walk_local_gate import WalkLocal
from monaqa2.mcmc.transition import create_transition_matrix
from monaqa2.mcmc.distribution import get_gibbs_distribution_hJ

n = 3
h = np.array([0.9, -0.6, 0.35])
J = np.array([
    [0.0, -0.75, 0.40],
    [-0.75, 0.0, -0.55],
    [0.40, -0.55, 0.0],
])
beta = 12.0
a = 10
c = 1

# Build the classical uniform-proposal transition matrix and its discriminant.
k = 1
P_local = create_proposal_matrix_local(n, k=k)
Q = create_transition_matrix(P_local, h, J, beta, a=a)
X = np.sqrt(Q * Q.T)

# Build the target Gibbs stationary state.
pi = get_gibbs_distribution_hJ(h, J, beta=beta)
sqrt_pi = np.sqrt(pi).reshape(-1, 1)

# Here mocked_circuit=True means we use the ancilla-free versions of Reflection and AcceptPath.
walk = WalkLocal(n, k, h, J, beta, eps=None, coin="stateprep", a=a, c=c, mocked_circuit=True)
qc = QuantumCircuit(walk.num_qubits)
qc.append(walk, range(walk.num_qubits))

# Extract the unitary matrix of the circuit.
W_local = np.real_if_close(get_unitary(qc, big_endian=True))

In [19]:
print("Check 1: is the walk W_local a block encoding for X_local?")
left_zero = kron(np.eye(2**n), bra(0, n), bra(0, walk.coins), bra(0, walk.n_accept_work), bra(0, walk.n_reflection_work))
right_zero = kron(np.eye(2**n), ket(0, n), ket(0, walk.coins), ket(0, walk.n_accept_work), ket(0, walk.n_reflection_work))
actual_X = np.real_if_close(left_zero @ W_local @ right_zero)
block_encoding_err = np.linalg.norm(X - actual_X, ord=2)
print(f"\tBlock-encoding error ||X - <0|W|0>|| = {block_encoding_err:1.4e}")
print(f"\tTest passed: {np.abs(block_encoding_err) <= 1e-6}")

print("\nCheck 2: is the coherent gibbs state |pi>|0>|0> a +1 eigenstate of W_local?")
stationary_state = kron(sqrt_pi, ket(0, n), ket(0, walk.coins), ket(0, walk.n_accept_work), ket(0, walk.n_reflection_work))
stationary_state_err = np.linalg.norm(W_local @ stationary_state - stationary_state)
print(f"\tWalk stationary-state error ||W|sqrt(pi),0,0> - |sqrt(pi),0,0>|| = {stationary_state_err:1.4e}")
print(f"\tTest passed: {np.abs(stationary_state_err) <= 1e-6}")

print("\nCheck 3: is each lambda in spec(X_local) associated with exp(+- i arccos(lambda)) in spec(W_local)?")
actual_walk_eigs = np.linalg.eigvals(W_local)
expected_walk_eigs = np.ravel([[np.exp(1j * np.arccos(lam)), np.exp(-1j * np.arccos(lam))] 
                               for lam in np.clip(np.linalg.eigvalsh(X), -1.0, 1.0)])
err_spectral = max([np.min(np.abs(actual_walk_eigs - eig)) for eig in expected_walk_eigs])
print(f"\tMax spectral-mapping error = {err_spectral:1.4e}")
print(f"\tTest passed: {np.abs(err_spectral) <= 1e-6}")

Check 1: is the walk W_local a block encoding for X_local?
	Block-encoding error ||X - <0|W|0>|| = 3.7893e-13
	Test passed: True

Check 2: is the coherent gibbs state |pi>|0>|0> a +1 eigenstate of W_local?
	Walk stationary-state error ||W|sqrt(pi),0,0> - |sqrt(pi),0,0>|| = 3.7832e-13
	Test passed: True

Check 3: is each lambda in spec(X_local) associated with exp(+- i arccos(lambda)) in spec(W_local)?
	Max spectral-mapping error = 1.4901e-08
	Test passed: True


## Walk with quantum-enhanced move

**Caution**: differently from the uniform and local moves, the QeMC proposal unitary does not directly encode the nonnegative proposal amplitudes $\sqrt{P_{\mathrm{qemc}}(y,x)}$. It encodes the complex Hamiltonian-evolution amplitudes $U_{yx}=\bra{y}e^{-iHt}\ket{x}$. Therefore, comparing the circuit amplitudes with $\sqrt{P_{\mathrm{qemc}}}$ is not the correct test. The compatibility with the Szegedy walk comes instead from the discriminant contraction, where phases cancel provided the implemented evolution is symmetric in the computational basis. 

This is exact for a real symmetric Hamiltonian with exact evolution; for product formulas, one should prefer a symmetric formula, such as second-order Trotter, if this symmetry is meant to hold at the circuit level.

### Proposal matrix

In [11]:
from monaqa2.qiskit.proposal_qemc_gate import ProposalQemc
from monaqa2.mcmc.proposal import create_proposal_matrix_quantum_exact

# Set the data to create the proposal matrix. This time the move knows about the ising model 
n = 3
gamma = 0.7
t = 1.0
h = np.array([0.9, -0.6, 0.35])
J = np.array([
    [0.0, -0.75, 0.40],
    [-0.75, 0.0, -0.55],
    [0.40, -0.55, 0.0],
])

# Build the quantum circuit implementing the local proposal.
v_qemc = ProposalQemc(n, h, J, gamma=gamma, t=t, evolution="exact")
qc = QuantumCircuit(v_qemc.num_qubits)
qc.append(v_qemc, range(v_qemc.num_qubits))

Unlike the previous moves, $V_{\mathrm{qemc}}$ does not directly encode the square root of the entries of $P_{\mathrm{qemc}}$. In fact,

$$
({}_a\bra{x}{}_b\bra{y}) \,
V_{\mathrm{qemc}} \,
(\ket{x}_a\ket{0}_b)
=
\bra{y} e^{-iHt} \ket{x}
=
U_{yx}
\neq
|U_{yx}|
=
\sqrt{P_{\mathrm{qemc}}(y,x)}.
$$

For uniform and local proposals, the amplitudes are real and nonnegative, so this distinction is invisible. For Hamiltonian evolution, the amplitudes generally have phases, so the desired condition does not hold at the level of amplitudes.


In [12]:
# Build the ideal proposal-amplitude matrix for exactly-k local moves.
P_qemc = create_proposal_matrix_quantum_exact(h, J, gamma=gamma, t=t)
expected = np.sqrt(P_qemc)

# Extract the unitary matrix of the circuit.
V_qemc = np.real_if_close(get_unitary(qc, big_endian=True))
actual = sum(kron(bra(x, n), np.eye(2**n)) @ V_qemc @ kron(ketbra(x, n), ket(0, n)) for x in range(2**n))

# Compare the ideal and circuit-derived proposal amplitudes.
err = np.linalg.norm(expected - actual, ord=2)
print(f"The error between the ideal transformation and the quantum circuit is {err:1.4e}")

The error between the ideal transformation and the quantum circuit is 3.4134e+00


To check the proposal matrix, one should instead compare probabilities: $P_{\mathrm{qemc}}(y,x)
= |U_{yx}|^2$. However, we cannot access the magnitude directly. 

In [13]:
actual = np.abs(actual)
err = np.linalg.norm(expected - actual, ord=2)
print(f"The error between the ideal transformation and the quantum circuit is {err:1.4e}")

The error between the ideal transformation and the quantum circuit is 0.0000e+00


### Walk operator

The QeMC proposal does not directly encode the nonnegative amplitudes $\sqrt{P_{\mathrm{qemc}}(y,x)}$. Instead, it coherently prepares the Hamiltonian-evolution amplitudes:

$$
({}_a\bra{x}{}_b\bra{y})\,
V_{\mathrm{qemc}}\,
(\ket{x}_a\ket{0}_b)
=
U_{yx},
\qquad
U=e^{-iHt}.
$$

Therefore, $V_{\mathrm{qemc}}$ alone gives access to $U_{yx}$, not to $|U_{yx}|=\sqrt{P_{\mathrm{qemc}}(y,x)}$. However, the phases cancel in the Szegedy discriminant contraction. If $H=H^\top$, then $U=e^{-iHt}$ is symmetric, and

$$
({}_a\bra{x}{}_b\bra{0})\,
V_{\mathrm{qemc}}^\dagger \; \mathrm{SWAP}_{a,b} \; V_{\mathrm{qemc}}\,
(\ket{y}_a\ket{0}_b)
=
U_{xy}^* U_{yx}
=
|U_{xy}|^2
=
P_{\mathrm{qemc}}(x,y).
$$

The actual walk also includes the acceptance coin. It is built from

$$
U_{\mathrm{acc}}
=
V_{\mathrm{qemc}}^\dagger B^\dagger F B V_{\mathrm{qemc}},
$$

where $B$ prepares the acceptance amplitude and $F$ swaps the two registers only along the accepted coin path. If $A_{yx}$ is the probability of accepting the move $x\to y$, then for $x\neq y$ the zero-block matrix element is

$$
({}_a\bra{x}{}_b\bra{0}{}_c\bra{0})\,
U_{\mathrm{acc}}\,
(\ket{y}_a\ket{0}_b\ket{0}_c)
=
U_{xy}^*U_{yx}\sqrt{A_{xy}A_{yx}}.
$$

Using again $U_{xy}=U_{yx}$, this becomes

$$
|U_{xy}|^2\sqrt{A_{xy}A_{yx}}
=
P_{\mathrm{qemc}}(x,y)\sqrt{A_{xy}A_{yx}}.
$$

Now define the accepted transition matrix $Q$ by $Q_{yx}=P_{\mathrm{qemc}}(y,x)A_{yx}$ for $y\neq x$, with the diagonal absorbing rejected moves. Since $P_{\mathrm{qemc}}$ is symmetric, the off-diagonal discriminant of $Q$ is

$$
X_Q(x,y)
=
\sqrt{Q_{xy}Q_{yx}}
=
P_{\mathrm{qemc}}(x,y)\sqrt{A_{xy}A_{yx}}.
$$

Thus,

$$
(\mathbb{I}_a\otimes\bra{0}_b\otimes\bra{0}_c)\,
V_{\mathrm{qemc}}^\dagger B^\dagger F B V_{\mathrm{qemc}}\,
(\mathbb{I}_a\otimes\ket{0}_b\otimes\ket{0}_c)
=
X_Q.
$$

Therefore, the walk implements the usual Szegedy spectral transformation of the discriminant $X_Q$: if $\lambda\in\mathrm{spec}(X_Q)$, then the corresponding walk eigenvalues are $e^{\pm i\arccos(\lambda)}$.

In [14]:
from monaqa2.qiskit.walk_qemc_gate import WalkQemc
from monaqa2.mcmc.transition import create_transition_matrix
from monaqa2.mcmc.distribution import get_gibbs_distribution_hJ

# Other parameters
beta = 12.0
a = 10
c = 1

# Build the classical uniform-proposal transition matrix and its discriminant.
Q = create_transition_matrix(P_qemc, h, J, beta, a=a)
X = np.sqrt(Q * Q.T)

# Build the target Gibbs stationary state.
pi = get_gibbs_distribution_hJ(h, J, beta=beta)
sqrt_pi = np.sqrt(pi).reshape(-1, 1)

# Here mocked_circuit=True means we use the ancilla-free versions of Reflection and AcceptPath.
walk = WalkQemc(n, h, J, gamma=gamma, beta=beta, t=t, eps=None, coin="stateprep", a=a, c=c, evolution="exact", mocked_circuit=True)
qc = QuantumCircuit(walk.num_qubits)
qc.append(walk, range(walk.num_qubits))

# Extract the unitary matrix of the circuit.
W_qemc = np.real_if_close(get_unitary(qc, big_endian=True))

In [15]:
print("Check 1: is the walk W_qemc a block encoding for X_qemc?")
left_zero = kron(np.eye(2**n), bra(0, n), bra(0, walk.coins), bra(0, walk.n_accept_work), bra(0, walk.n_reflection_work))
right_zero = kron(np.eye(2**n), ket(0, n), ket(0, walk.coins), ket(0, walk.n_accept_work), ket(0, walk.n_reflection_work))
actual_X = np.real_if_close(left_zero @ W_qemc @ right_zero)
block_encoding_err = np.linalg.norm(X - actual_X, ord=2)
print(f"\tBlock-encoding error ||X - <0|W|0>|| = {block_encoding_err:1.4e}")
print(f"\tTest passed: {np.abs(block_encoding_err) <= 1e-6}")

print("\nCheck 2: is the coherent gibbs state |pi>|0>|0> a +1 eigenstate of W_qemc?")
stationary_state = kron(sqrt_pi, ket(0, n), ket(0, walk.coins), ket(0, walk.n_accept_work), ket(0, walk.n_reflection_work))
stationary_state_err = np.linalg.norm(W_qemc @ stationary_state - stationary_state)
print(f"\tWalk stationary-state error ||W|sqrt(pi),0,0> - |sqrt(pi),0,0>|| = {stationary_state_err:1.4e}")
print(f"\tTest passed: {np.abs(stationary_state_err) <= 1e-6}")

print("\nCheck 3: is each lambda in spec(X_qemc) associated with exp(+- i arccos(lambda)) in spec(W_qemc)?")
actual_walk_eigs = np.linalg.eigvals(W_qemc)
expected_walk_eigs = np.ravel([[np.exp(1j * np.arccos(lam)), np.exp(-1j * np.arccos(lam))] 
                               for lam in np.clip(np.linalg.eigvalsh(X), -1.0, 1.0)])
err_spectral = max([np.min(np.abs(actual_walk_eigs - eig)) for eig in expected_walk_eigs])
print(f"\tMax spectral-mapping error = {err_spectral:1.4e}")
print(f"\tTest passed: {np.abs(err_spectral) <= 1e-6}")

Check 1: is the walk W_qemc a block encoding for X_qemc?
	Block-encoding error ||X - <0|W|0>|| = 3.4064e-13
	Test passed: True

Check 2: is the coherent gibbs state |pi>|0>|0> a +1 eigenstate of W_qemc?
	Walk stationary-state error ||W|sqrt(pi),0,0> - |sqrt(pi),0,0>|| = 3.4084e-13
	Test passed: True

Check 3: is each lambda in spec(X_qemc) associated with exp(+- i arccos(lambda)) in spec(W_qemc)?
	Max spectral-mapping error = 3.4829e-13
	Test passed: True
